# Round37 Full Workflow Notebook: `v37_5_ta40_ec40_push.csv`

这个 notebook 展示了后期突破阶段的完整流程：从官方 GitHub 原始数据读取，到 Hydro corridor 共识增量构造、导出提交文件、再到与公开参考文件逐列核对。

**为什么这本 notebook 重要**
- 这是项目第一次把官方分数推到 `0.376` 的关键阶段。
- 策略核心是把 `TA` 和 `EC` 的正向安全增量同时推到 `0.40 / 0.40`。
- DRP 在这个阶段保持锚定，避免破坏整体稳定性。


## Step 1. Experiment Objective

**这个步骤做什么**
说明 round37 突破分支的策略目标和设计原则。

**为什么要做这个步骤**
后期 notebook 不能只给结果，还要讲清楚为什么这一步值得测试。

**预期产出**
得到本分支的实验角色定义。


**本分支摘要**
- Output file: `v37_5_ta40_ec40_push.csv`
- Strategy family: `Hydro micro corridor`
- TA alpha: `0.40`
- EC alpha: `0.40`
- DRP source: `legacy control`


## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并定义官方数据与公开资产地址。

**为什么要做这个步骤**
保证 notebook 从第一行开始就能看懂输入数据来自哪里。

**预期产出**
得到统一运行环境和 URL 常量。


In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖，定义官方数据源与公开 repo 资产地址
# =========================

from pathlib import Path
from io import StringIO
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 220)

OFFICIAL_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'
SHOWCASE_RAW = 'https://raw.githubusercontent.com/FlalaGoGoGo/ey-water-2026-showcase/main'
OUTPUT_DIR = Path('generated_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


def read_csv_remote(url: str) -> pd.DataFrame:
    """优先 pandas 直读，失败时回退到 curl，保证 notebook 更稳。"""
    try:
        return pd.read_csv(url)
    except Exception as exc:
        print(f'[warn] pandas direct read failed: {url}')
        print(f'       reason: {exc}')
        res = subprocess.run(['curl', '-L', url], check=True, capture_output=True, text=True)
        return pd.read_csv(StringIO(res.stdout))


## Step 3. Load Official Data And Public Anchors

**这个步骤做什么**
读取官方验证模板、训练数据，以及公开 repo 中保存的关键锚点提交文件。

**为什么要做这个步骤**
这个阶段的策略不是黑盒模型，而是围绕已验证安全锚点做结构化增量。

**预期产出**
得到构造 round37 提交所需的全部输入表。


In [ ]:
# 官方原始数据
train_wq = read_csv_remote(f'{OFFICIAL_RAW}/water_quality_training_dataset.csv')
val_tpl = read_csv_remote(f'{OFFICIAL_RAW}/submission_template.csv')

# 公开 repo 中保存的策略锚点
legacy_control = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/anchors/round29/v29_5_static_hydro_ood_guard.csv')
control_30 = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/anchors/round36/v36_1_control_v35_1.csv')
reference_target = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/targets/v37_5_ta40_ec40_push.csv')

print('train_wq:', train_wq.shape)
print('val_tpl:', val_tpl.shape)
print('legacy_control:', legacy_control.shape)
print('control_30:', control_30.shape)
print('reference_target:', reference_target.shape)


## Step 4. Quick QA And Context

**这个步骤做什么**
检查日期范围、站点数量、以及控制锚点与官方模板是否对齐。

**为什么要做这个步骤**
先确认输入表结构正确，后面的差分和渲染才有意义。

**预期产出**
得到一个可审计的 round37 输入上下文。


In [ ]:
train_wq['Sample Date'] = pd.to_datetime(train_wq['Sample Date'], dayfirst=True, errors='coerce')
val_dates = pd.to_datetime(val_tpl['Sample Date'], dayfirst=True, errors='coerce')

print('training date range:', train_wq['Sample Date'].min(), '->', train_wq['Sample Date'].max())
print('validation date range:', val_dates.min(), '->', val_dates.max())
print('unique training sites:', train_wq[['Latitude', 'Longitude']].drop_duplicates().shape[0])
print('template rows:', len(val_tpl))

display(train_wq.head(3))
display(legacy_control.head(3))


## Step 5. Build Consensus Deltas

**这个步骤做什么**
把 legacy control 和 round36 control 的差值转换成可控的 TA/EC 共识增量。

**为什么要做这个步骤**
round37 的核心不是重新训练，而是沿着已经在线验证过的正向方向继续推。

**预期产出**
得到可用于 outer-boundary push 的 TA/EC 增量数组。


In [ ]:
base_ta = legacy_control['Total Alkalinity'].to_numpy(dtype=float)
base_ec = legacy_control['Electrical Conductance'].to_numpy(dtype=float)
base_drp = legacy_control['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

control_ta = control_30['Total Alkalinity'].to_numpy(dtype=float)
control_ec = control_30['Electrical Conductance'].to_numpy(dtype=float)

# 0.30 是上一阶段 control 的安全推进强度，这里把它还原成单位共识增量。
ta_consensus_delta = np.maximum(control_ta - base_ta, 0.0) / 0.30
ec_consensus_delta = np.maximum(control_ec - base_ec, 0.0) / 0.30

print('mean ta consensus delta:', round(float(np.mean(ta_consensus_delta)), 6))
print('mean ec consensus delta:', round(float(np.mean(ec_consensus_delta)), 6))


## Step 6. Render The 40/40 Push

**这个步骤做什么**
按 round37 的配置把 TA 和 EC 同时推到 0.40，DRP 保持 legacy 锚点。

**为什么要做这个步骤**
这个版本的目标是验证联合边界 40/40 是否还能在线保持安全。

**预期产出**
得到最终预测数组。


In [ ]:
ta = np.clip(base_ta + 0.40 * ta_consensus_delta, 0, None)
ec = np.clip(base_ec + 0.40 * ec_consensus_delta, 0, None)
drp = np.clip(base_drp, 0, None)

submission = val_tpl[['Latitude', 'Longitude', 'Sample Date']].copy()
submission['Sample Date'] = pd.to_datetime(submission['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d/%m/%Y')
submission['Total Alkalinity'] = ta
submission['Electrical Conductance'] = ec
submission['Dissolved Reactive Phosphorus'] = drp
submission = submission[['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]
display(submission.head(3))


## Step 7. Export And Verify

**这个步骤做什么**
导出 round37 文件，并和公开参考结果逐列比较。

**为什么要做这个步骤**
公开 notebook 最好不仅能生成文件，还能证明自己复现的是正确版本。

**预期产出**
得到导出 csv 与 exact-match 诊断。


In [ ]:
out_path = OUTPUT_DIR / 'v37_5_ta40_ec40_push.csv'
submission.to_csv(out_path, index=False)
print('saved to:', out_path.resolve())

metric_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
diff = (submission[metric_cols] - reference_target[metric_cols]).abs()
print('max abs diff by target:')
print(diff.max())
print('exact match:', bool((diff.max() < 1e-12).all()))


## Step 8. Diagnostic Review

**这个步骤做什么**
用简单图表查看 round37 相对 legacy control 的整体位移。

**为什么要做这个步骤**
这一步帮助外部读者理解为什么这个版本属于“结构化边界推进”而不是随机改数。

**预期产出**
得到 TA/EC/DRP 三个目标的整体位移解释。


In [ ]:
shift_summary = pd.DataFrame({
    'target': ['TA', 'EC', 'DRP'],
    'mean_abs_shift': [
        float(np.mean(np.abs(submission['Total Alkalinity'] - legacy_control['Total Alkalinity']))),
        float(np.mean(np.abs(submission['Electrical Conductance'] - legacy_control['Electrical Conductance']))),
        float(np.mean(np.abs(submission['Dissolved Reactive Phosphorus'] - legacy_control['Dissolved Reactive Phosphorus']))),
    ]
})
display(shift_summary)

plt.figure(figsize=(7, 4))
plt.bar(shift_summary['target'], shift_summary['mean_abs_shift'], color=['#ffe600', '#ffd166', '#9cd18b'])
plt.title('Round37 mean absolute shift vs legacy control')
plt.ylabel('Mean absolute shift')
plt.show()


## Step 9. Interpretation

**这个步骤做什么**
把 round37 的结果放回到比赛节奏里解释。

**为什么要做这个步骤**
公开 notebook 不应该只有代码，还要告诉外部读者这一版为什么重要。

**本分支结论**
- TA 和 EC 同时推进到 `0.40 / 0.40` 后，round37 首次到达 `0.376`。
- DRP 在这一阶段继续保持锚定，说明主收益来自 TA/EC corridor。
- 这本 notebook 是后期平台段所有 tied frontier 分支的起点。
